In [1]:
"""
This script tests samples periodically from a bioreactor. 
The samples are dispensed into a BioER Deepwell plate containing buffer like arresting buffer
2025-10-15
Steps:
1) Begin bioreactor experiment; change bioreactor name and experiment below. Bring to temp.
2) Make deepwell plate. Place 1mL Arresting buffer in rows A and E. 
3) Change sampling time below. 30 min? Begin script

2025-11-15
adding await to functions. 
Need to confirm that OD reading esp Normalized OD readings aren't being cancelled.

2026-02-05
To run do the following:
1)change sampling time below
2)confirm pioreactor e.g. leadere1
3)begin pioreactor experiment with stirring, temp, OD and copy experiment name below
4)make sure 50ul filtered tips in position=1, not 0, and full
5)add 150ul fresh culture to tube and then begin growth rate on pioreactor
6)begin growth curve on pioreactor
7)then begin this program (don't forget to switch git branches if needed, "git checkout add-pioreactor")

"""




'\nThis script tests samples periodically from a bioreactor. \nThe samples are dispensed into a BioER Deepwell plate containing buffer like arresting buffer\n2025-10-15\nSteps:\n1) Begin bioreactor experiment; change bioreactor name and experiment below. Bring to temp.\n2) Make deepwell plate. Place 1mL Arresting buffer in rows A and E. \n3) Change sampling time below. 30 min? Begin script\n\n2025-11-15\nadding await to functions. \nNeed to confirm that OD reading esp Normalized OD readings aren\'t being cancelled.\n\n2026-02-05\nTo run do the following:\n1)change sampling time below\n2)confirm pioreactor e.g. leadere1\n3)begin pioreactor experiment with stirring, temp, OD and copy experiment name below\n4)make sure 50ul filtered tips in position=1, not 0, and full\n5)add 150ul fresh culture to tube and then begin growth rate on pioreactor\n6)begin growth curve on pioreactor\n7)then begin this program (don\'t forget to switch git branches if needed, "git checkout add-pioreactor")\n\n'

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:

import asyncio
from datetime import datetime, timedelta, timezone

from pylabrobot.liquid_handling import LiquidHandler
from pylabrobot.liquid_handling.backends import STARBackend
from pylabrobot.resources.hamilton import STARLetDeck, MFX_CAR_L5_base, TIP_CAR_480_A00
from pylabrobot.resources.hamilton.mfx_modules import Hamilton_MFX_plateholder_DWP_metal_tapped
from pylabrobot.resources.bioer.plates import BioER_96_wellplate_Vb_2200uL
# from pylabrobot.resources.bioer.plates import BioER_96_wellplate_Vb_2200uL
from pylabrobot.resources.diy.grindbio.modules import Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint
from pylabrobot.resources.pioreactor import pioreactor
from pylabrobot.resources import (
 hamilton_96_tiprack_50uL_filter, # 50 µL filtered
    hamilton_96_tiprack_1000uL, # 1000 µL filtered
    hamilton_96_tiprack_10uL_filter, # 10 µL filtered
)
from pylabrobot.resources import Coordinate

###############################################################################
# 0) Build LiquidHandler + deck
###############################################################################
backend = STARBackend()
lh = LiquidHandler(backend=backend, deck=STARLetDeck())
await lh.setup(skip_autoload=True)

2026-02-13 19:16:29,031 - pylabrobot.io.usb - INFO - Finding USB device...
2026-02-13 19:16:29,085 - pylabrobot.io.usb - INFO - Found USB device.
2026-02-13 19:16:29,088 - pylabrobot.io.usb - INFO - Found endpoints. 
Write:
       ENDPOINT 0x2: Bulk OUT ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :    0x2 OUT
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0 
Read:
       ENDPOINT 0x81: Bulk IN ===============================
       bLength          :    0x7 (7 bytes)
       bDescriptorType  :    0x5 Endpoint
       bEndpointAddress :   0x81 IN
       bmAttributes     :    0x2 Bulk
       wMaxPacketSize   :   0x40 (64 bytes)
       bInterval        :    0x0
2026-02-13 19:16:32,262 - pylabrobot - INFO - Running backend initialization procedure.


In [4]:

###############################################################################
# 1) Carriers, modules & labware
###############################################################################
# --- tip carrier -------------------------------------------------------------
tip_car = TIP_CAR_480_A00("tip_car")
lh.deck.assign_child_resource(tip_car, rails=25)

tiprack_1000 = hamilton_96_tiprack_1000uL("tips_00")               # 1000 µL filter tips (slot-0)
tiprack_50   = hamilton_96_tiprack_50uL_filter("tips_01")  #  50 µL filter tips (slot-1)
tiprack_10   = hamilton_96_tiprack_10uL_filter("tips_02")                #  10 µL filter tips (slot-2)

# Mount the racks
tip_car[0] = tiprack_1000
tip_car[1] = tiprack_50
tip_car[2] = tiprack_10

# --- deep-well plate on MFX holder ------------------------------------------
module0 = Hamilton_MFX_plateholder_DWP_metal_tapped("module0")
car_13 = MFX_CAR_L5_base("car_13", modules={0: module0})
lh.deck.assign_child_resource(car_13, rails=13)

dwplate = BioER_96_wellplate_Vb_2200uL("dwPlate")
module0.assign_child_resource(dwplate)

# --- pioreactor on the 10mm raised holder -----------------------------------
moduleMod = Hamilton_MFX_plateholder_DWP_metal_tapped_10mm_3dprint("moduleMod")
car_07 = MFX_CAR_L5_base("car_07", modules={0: moduleMod})
lh.deck.assign_child_resource(car_07, rails=7)

pr = pioreactor("pr")
moduleMod.assign_child_resource(pr)

/tmp/ipykernel_52192/820125636.py:18: DeprecationWarning: Hamilton_MFX_plateholder_DWP_metal_tapped is deprecated. Use 'hamilton_mfx_plateholder_DWP_metal_tapped' instead.
  module0 = Hamilton_MFX_plateholder_DWP_metal_tapped("module0")
/tmp/ipykernel_52192/820125636.py:19: DeprecationWarning: MFX_CAR_L5_base is deprecated. Use 'hamilton_mfx_carrier_L5_base' instead.
  car_13 = MFX_CAR_L5_base("car_13", modules={0: module0})
/tmp/ipykernel_52192/820125636.py:27: DeprecationWarning: MFX_CAR_L5_base is deprecated. Use 'hamilton_mfx_carrier_L5_base' instead.
  car_07 = MFX_CAR_L5_base("car_07", modules={0: moduleMod})
2026-02-13 19:17:08,219 - pylabrobot - WARNING - Resource 'pr' is very high on the deck: 259.955 mm. Be careful when traversing the deck.
2026-02-13 19:17:08,220 - pylabrobot - WARNING - Resource 'pr_well_A1' is very high on the deck: 266.455 mm. Be careful when traversing the deck.


In [5]:
##############################################################################
# OD-reading helpers (continuous + snapshot) and small job utilities
###############################################################################
import time
import requests
from urllib.parse import quote

import time
import requests
from urllib.parse import quote

# BASE = "http://leadera1.local"
BASE = "http://leadere1.local"
UNIT = "leaderE1"
# EXP  = "sample_and_aliquot_od"
# EXP  = "slowerstir_for_better_OD"
EXP  = "Jared6_growth_aliquots"

async def run_stirring(rpm=500):
    url = f"{BASE}/api/workers/{UNIT}/jobs/run/job_name/stirring/experiments/{quote(EXP)}"
    resp = requests.patch(url, json={"options": {"target_rpm": rpm}},
                          headers={"Content-Type": "application/json"})
    print("START:", resp.status_code, resp.text)
    return resp

async def list_running():
    url = f"{BASE}/api/workers/{UNIT}/jobs/running"
    r = requests.get(url)
    try:
        txt = r.text
    except Exception:
        txt = ""
    print("RUNNING:", r.status_code, txt)
    return r

async def wait_until_stopped(job_name="stirring", timeout_s=15, poll_s=0.5):
    """Poll /jobs/running until job_name disappears or timeout."""
    deadline = time.time() + timeout_s
    url = f"{BASE}/api/workers/{UNIT}/jobs/running"
    while time.time() < deadline:
        r = requests.get(url)
        if r.ok and (job_name not in r.text):
            return True
        time.sleep(poll_s)
    return False

async def stop_stirring():
    # Preferred “stop this specific job” endpoint
    stop_specific = f"{BASE}/api/workers/{UNIT}/jobs/stop/job_name/stirring/experiments/{quote(EXP)}"
    resp = requests.patch(stop_specific, headers={"Content-Type": "application/json"})
    print("STOP specific:", resp.status_code, getattr(resp, "text", ""))

    if await wait_until_stopped(job_name="stirring"):
        print("Confirmed: stirring stopped.")
        return True

    # Fallback #1: stop ALL jobs for this unit in this experiment
    stop_all = f"{BASE}/api/workers/{UNIT}/jobs/stop/experiments/{quote(EXP)}"
    resp2 = requests.patch(stop_all, headers={"Content-Type": "application/json"})
    print("STOP all-in-exp:", resp2.status_code, getattr(resp2, "text", ""))

    if await wait_until_stopped(job_name="stirring"):
        print("Confirmed after stop-all: stirring stopped.")
        return True

    # Fallback #2: unit_api stop (query params)
    unit_api_stop = f"{BASE}/unit_api/jobs/stop?job_name=stirring&experiment={quote(EXP)}"
    resp3 = requests.patch(unit_api_stop, headers={"Content-Type": "application/json"})
    print("STOP unit_api:", resp3.status_code, getattr(resp3, "text", ""))

    ok = await wait_until_stopped(job_name="stirring")
    print("Final stop status:", "stopped" if ok else "still running")
    return ok

async def update_stirring(rpm):
    url = f"{BASE}/api/workers/{UNIT}/jobs/update/job_name/stirring/experiments/{quote(EXP)}"
    resp = requests.patch(url, json={"settings": {"target_rpm": rpm}},
                          headers={"Content-Type": "application/json"})
    print("UPDATE:", resp.status_code, resp.text)
    return resp

async def is_job_running(job_name="od_reading"):
    url = f"{BASE}/api/workers/{UNIT}/jobs/running"
    try:
        r = requests.get(url, headers={"Content-Type": "application/json"})
        if r.ok:
            return job_name in r.text
    except Exception:
        pass
    return False

async def wait_until_stopped(job_name="stirring", timeout_s=15, poll_s=0.5):
    """Poll /jobs/running until job_name disappears or timeout."""
    deadline = time.time() + timeout_s
    url = f"{BASE}/api/workers/{UNIT}/jobs/running"
    while time.time() < deadline:
        r = requests.get(url)
        if r.ok and (job_name not in r.text):
            return True
        time.sleep(poll_s)
    return False

async def stop_od_reading():
    """Stop od_reading with the same multi-tier approach as stirring."""
    stop_specific = f"{BASE}/api/workers/{UNIT}/jobs/stop/job_name/od_reading/experiments/{quote(EXP)}"
    r1 = requests.patch(stop_specific, headers={"Content-Type": "application/json"})
    print("STOP od_reading specific:", r1.status_code, getattr(r1, "text", ""))

    if wait_until_stopped(job_name="od_reading"):
        print("Confirmed: od_reading stopped.")
        return True


    # Fallback: stop all jobs in this experiment
    stop_all = f"{BASE}/api/workers/{UNIT}/jobs/stop/experiments/{quote(EXP)}"
    r2 = requests.patch(stop_all, headers={"Content-Type": "application/json"})
    print("STOP all-in-exp (od_reading fallback):", r2.status_code, getattr(r2, "text", ""))

    if wait_until_stopped(job_name="od_reading"):
        print("Confirmed after stop-all: od_reading stopped.")
        return True

    # Fallback 2: unit_api query-params
    unit_api_stop = f"{BASE}/unit_api/jobs/stop?job_name=od_reading&experiment={quote(EXP)}"
    r3 = requests.patch(unit_api_stop, headers={"Content-Type": "application/json"})
    print("STOP unit_api od_reading:", r3.status_code, getattr(r3, "text", ""))

    ok = wait_until_stopped(job_name="od_reading")
    print("Final od_reading stop status:", "stopped" if ok else "still running")
    return ok

async def run_od_continuous():
    """Start od_reading (continuous). Idempotent if already running."""
    if await is_job_running("od_reading"):
        print("od_reading already running (continuous).")
        return None
    run_url = f"{BASE}/api/workers/{UNIT}/jobs/run/job_name/od_reading/experiments/{quote(EXP)}"
    # No snapshot -> continuous according to job defaults
    resp = requests.patch(run_url, json={"options": {}}, headers={"Content-Type": "application/json"})
    print("RUN od_reading (continuous):", resp.status_code, getattr(resp, "text", ""))
    return resp

async def run_od_snapshot(timeout_s=20):
    """
    Start od_reading with snapshot=True so it takes one reading and exits.
    Assumes od_reading is NOT already running.
    """
    run_url = f"{BASE}/api/workers/{UNIT}/jobs/run/job_name/od_reading/experiments/{quote(EXP)}"
    payload = {"options": {"snapshot": True}}
    resp = requests.patch(run_url, json=payload, headers={"Content-Type": "application/json"})
    print("OD snapshot run:", resp.status_code, getattr(resp, "text", ""))

    done = wait_until_stopped(job_name="od_reading", timeout_s=timeout_s, poll_s=0.5)
    print("OD snapshot status:", "completed" if done else "still running / timed out")
    return resp, done

# Log aliquot event to Pioreactor logs
async def log_the_aliquot_in_the_pioreactor_logs(cycle: int | None = None,
                                                 level: str = "INFO",
                                                 source: str = "automation",
                                                 task: str = ""):
    try:
        cyc = cycle
        # Compose message exactly as requested
        msg = f"Aliquot from Pioreactor at cycle {cyc}" if cyc is not None else "Aliquot from Pioreactor (cycle unknown)"

        url = f"{BASE}/api/workers/{UNIT}/experiments/{quote(EXP)}/logs"
        payload = {
            "experiment": EXP,
            "level": level,
            "message": msg,
            "pioreactor_unit": UNIT,
            "source": source,       # e.g., "automation" or "UI"
            "task": task,           # empty string is fine
            # Match GUI format: ISO8601 in UTC with 'Z'
            "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z"),
        }
        resp = requests.post(url, json=payload, headers={"Content-Type": "application/json"})
        print("LOG POST:", resp.status_code, getattr(resp, "text", ""))
        return resp
    except Exception as e:
        print("LOG POST ERROR:", repr(e))
        return None

In [6]:

###############################################################################
# 2) Parameters
###############################################################################
CYCLES = 11
TIME_BETWEEN_SAMPLING_MIN = 1  # minutes
CHANNEL_MM = 2                  # use channel 6 for the run
TIPRACK_50 = tiprack_50         # convenience alias
SAFE_Z = 270.0                  # mm above deck; should exceed tallest stack

###############################################################################
# 3) Helpers
###############################################################################
async def countdown(seconds: int, prefix: str = ""):
  """Simple terminal countdown (updates ~1/s)."""
  end = datetime.now() + timedelta(seconds=seconds)
  while True:
    remaining = int((end - datetime.now()).total_seconds())
    if remaining <= 0:
      print(f"\r{prefix}00:00 remaining. Starting next cycle...       ")
      break
    mins, secs = divmod(remaining, 60)
    print(f"\r{prefix}{mins:02d}:{secs:02d} remaining (ETA {end.strftime('%H:%M:%S')})   ", end="")
    await asyncio.sleep(1)
  # Move to next line after countdown
  print()

###############################################################################
# 4) Core action
###############################################################################
async def sample_and_aliquot(col: int):
  
  """Pick up tip, aspirate from pioreactor, dispense into two wells, discard tip."""
  try:
    # Tips: row A, moving across columns
    await lh.pick_up_tips(TIPRACK_50[f"A{col}"], use_channels=[CHANNEL_MM])
    
    # Aspirate from pioreactor (hover above crossbar a bit)
    await lh.aspirate(
      pr["A1"],
      vols=[45],
      use_channels=[CHANNEL_MM],
      transport_air_volume=[0],
      # pre_wetting_volume=[45], # this doesn't work with 50ul filtered tips
      settling_time=[2],
      liquid_height = [8], # above crossbar
      # offsets=[Coordinate(z=8.0)],  # ~8 mm above crossbar
      minimum_traverse_height_at_beginning_of_a_command=SAFE_Z,
      min_z_endpos=SAFE_Z,
    )

    # Dispense 20 µL into A{col}
    await lh.dispense(
      dwplate[f"A{col}"],
      vols=[20],
      use_channels=[CHANNEL_MM],
      liquid_height=[20],  # ~1 mL buffer preloaded in DW well
      settling_time=[1],
      transport_air_volume=[0],
      minimum_traverse_height_at_beginning_of_a_command=SAFE_Z,
      min_z_endpos=SAFE_Z,
    )

    # Dispense 20 µL into E{col}
    await lh.dispense(
      dwplate[f"E{col}"],
      vols=[20],
      use_channels=[CHANNEL_MM],
      liquid_height=[20],  # ~1 mL buffer preloaded
      settling_time=[1],
      transport_air_volume=[0],
      minimum_traverse_height_at_beginning_of_a_command=SAFE_Z,
      min_z_endpos=SAFE_Z,
    )
  finally:
    # Always try to discard tips to keep state sane
    try:
      await lh.discard_tips(use_channels=[CHANNEL_MM])
    except Exception:
      pass



In [7]:
# await run_stirring(rpm=500)
# await stop_stirring()
# from datetime import timezone
# await log_the_aliquot_in_the_pioreactor_logs(cycle=2)
# await stop_od_reading()     # start continuous OD
# await run_od_continuous()     # start continuous OD


In [8]:
###############################################################################
# 5) Run cycles with status UI inside the loop
###############################################################################
# Start continuous stirring + od_reading (they’ll be paused briefly at sampling)
run_stirring(600)
run_od_continuous()

from datetime import datetime, timedelta

start_time = datetime.now()
overall_eta = start_time + timedelta(minutes=TIME_BETWEEN_SAMPLING_MIN * (CYCLES - 1))
print(f"Run started at {start_time.strftime('%H:%M:%S')}. Estimated completion ~ {overall_eta.strftime('%H:%M:%S')}.")

for c in range(1, CYCLES + 1):
    cycle_start = datetime.now()
    exp_end = cycle_start + timedelta(minutes=TIME_BETWEEN_SAMPLING_MIN) if c < CYCLES else None
    print(f"\nCycle {c}/{CYCLES} — begin {cycle_start.strftime('%H:%M:%S')}"
          + (f", next cycle ETA {exp_end.strftime('%H:%M:%S')}" if exp_end else ", final cycle"))

    # === Pause background jobs prior to sampling ===
    # Stop stirring first (settles culture), then stop continuous od_reading.
    stopped_stir = await stop_stirring()
    time.sleep(3)
    if not stopped_stir:
        await update_stirring(0)
        time.sleep(3)
        print("Retrying stir stop after RPM=0 …")
        await stop_stirring()

    # Ensure od_reading is not running during the sampling window
    await stop_od_reading()

    # Optional settle period before the snapshot (reduces residual motion)
    time.sleep(2)

    # === Single-shot OD reading (equivalent to `pio run od_reading --snapshot`) ===
    await run_od_snapshot()

    # Small buffer to guarantee no background activity before sampling
    time.sleep(1)

    # === Perform sampling / aliquoting (no stirring, no od_reading running) ===
    # Use columns 1..12 then wrap if needed (simple example)
    time.sleep(2)  # chill before aliquoting if you want a short extra settle
    col = ((c - 1) % 12) + 1
    await sample_and_aliquot(col)
    await log_the_aliquot_in_the_pioreactor_logs(cycle=c)

    # === Resume continuous background jobs after sampling ===
    await run_od_continuous()     # restart continuous OD
    await run_stirring(600)       # resume stirring

    # === Countdown visualizer (between cycles only) ===
    if c < CYCLES:
        wait_seconds = int(TIME_BETWEEN_SAMPLING_MIN * 60)
        await countdown(wait_seconds, prefix=f"Waiting to start cycle {c+1}/{CYCLES}: ")

###############################################################################
# 6) After all cycles, keep both od_reading + stirring running
###############################################################################
await run_od_continuous()
await run_stirring(600)
print("All cycles complete. Continuous od_reading + stirring left running.")

/tmp/ipykernel_52192/540599357.py:5: RuntimeWarning: coroutine 'run_stirring' was never awaited
  run_stirring(600)
/tmp/ipykernel_52192/540599357.py:6: RuntimeWarning: coroutine 'run_od_continuous' was never awaited
  run_od_continuous()


Run started at 19:17:08. Estimated completion ~ 19:27:08.

Cycle 1/11 — begin 19:17:08, next cycle ETA 19:18:08
STOP specific: 202 {"status":"success"}
Confirmed: stirring stopped.
STOP od_reading specific: 202 {"status":"success"}
Confirmed: od_reading stopped.


/tmp/ipykernel_52192/932278527.py:109: RuntimeWarning: coroutine 'wait_until_stopped' was never awaited
  if wait_until_stopped(job_name="od_reading"):


OD snapshot run: 404 {"error":"Worker(s) leaderE1 not found, not active, or not assigned to experiment Jared6_growth_aliquots."}
OD snapshot status: completed


/tmp/ipykernel_52192/540599357.py:37: RuntimeWarning: coroutine 'wait_until_stopped' was never awaited
  await run_od_snapshot()


LOG POST: 202 {"status":"success"}
RUN od_reading (continuous): 404 {"error":"Worker(s) leaderE1 not found, not active, or not assigned to experiment Jared6_growth_aliquots."}
START: 404 {"error":"Worker(s) leaderE1 not found, not active, or not assigned to experiment Jared6_growth_aliquots."}
Waiting to start cycle 2/11: 00:59 remaining (ETA 19:18:48)   

CancelledError: 

In [ ]:
# await lh.dispense(trough["A1"], vols=[900], liquid_height=[2], use_channels=[CHANNEL_WATER])
# await lh.dispense(SRC_MM_TUBE, vols=[100], liquid_height=[4], use_channels=[CHANNEL_MM], blow_out=[1])
# await lh.dispense(pr7["A1"], vols=[50], use_channels=[CHANNEL_MM], blow_out=[1])
# await lh.drop_tips(TIPRACK_50["A1"], use_channels=[6])
# await lh.discard_tips()
# await lh.stop()
# await backend.stop() 
# # # Aspirate from pioreactor (hover above crossbar a bit)
# await lh.aspirate(
# pr["A1"],
# vols=[45],
# use_channels=[CHANNEL_MM],
# transport_air_volume=[0],
# # pre_wetting_volume=[45], # this doesn't work with 50ul filtered tips
# mix_volume = [50],
# mix_cycles = [2],
# settling_time=[2],
# liquid_height = [8], # above crossbar
# flow_rates=[20],
# # offsets=[Coordinate(z=8.0)],  # ~8 mm above crossbar
# minimum_traverse_height_at_beginning_of_a_command=SAFE_Z,
# min_z_endpos=SAFE_Z,
# )

# # Dispense 20 µL into E{col}
# await lh.dispense(
# dwplate["E1"],
# vols=[45],
# use_channels=[CHANNEL_MM],
# liquid_height=[20],  # ~1 mL buffer preloaded
# settling_time=[1],
# transport_air_volume=[0],
# minimum_traverse_height_at_beginning_of_a_command=SAFE_Z,
# min_z_endpos=SAFE_Z,
# )